In [2]:
import os
from datasets import load_dataset

print(os.getcwd())
print(os.listdir())

dataset = load_dataset("csv", data_files="./support_tickets_balanced_800.csv")

D:\ticker triager\ticket_triager\ml
['.ipynb_checkpoints', 'csecond.ipynb', 'first.ipynb', 'inference.py', 'new_version.ipynb', 'support_tickets.csv', 'support_tickets_400.csv', 'support_tickets_balanced_800.csv', 'synthetic_data.py', 'tickets_test.csv', 'tickets_train.csv', 'ticket_lora_adapter', 'urgency_lora_adapter', '__pycache__']


Generating train split: 0 examples [00:00, ? examples/s]

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['subject', 'body', 'category', 'urgency'],
        num_rows: 800
    })
})

In [4]:
import huggingface_hub

print(huggingface_hub.__version__)

1.30.0


In [5]:
import pandas as pd
df = pd.read_csv("support_tickets_balanced_800.csv")
df.head()

,subject,body,category,urgency
0,App crashes on launch,"Hi, the app closes every time I try to open it...",technical,low
1,File upload fails,"Good day, uploads fail even when I use support...",technical,low
2,Cannot access old account,"Good day, I no longer have access to the email...",account,high
3,Plan downgrade not applied,"Dear customer support, my requested plan downg...",subscription,medium
4,Account locked,"Hi, my account was locked after several login ...",account,high


In [6]:
df.iloc[0].body

'Hi, the app closes every time I try to open it. I can share the relevant order or account details privately. The issue is intermittent, so guidance would be helpful. Thank you for your assistance.'

In [7]:
from huggingface_hub import model_info

info = model_info("distilbert/distilbert-base-uncased")
print(info.id)

distilbert/distilbert-base-uncased


In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8)  # 8 categories

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    modules_to_save=["classifier", "pre_classifier"],  # <-- add this line
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 744,200 || all params: 67,703,824 || trainable%: 1.0992


In [10]:
# Define your label mapping once
label_list = ["billing", "account", "orders", "shipping", "refund", "technical", "product", "subscription"]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

def tokenize(batch):
    texts = [s + " " + b for s, b in zip(batch["subject"], batch["body"])]
    encoded = tokenizer(texts, truncation=True, padding="max_length")
    encoded["labels"] = [label2id[cat] for cat in batch["category"]]
    return encoded

tokenized = dataset.map(tokenize, batched=True)

from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
)

trainer = Trainer(model=model, args=args, train_dataset=tokenized["train"])
trainer.train()

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.089134
20,2.016048
30,1.978612
40,1.936750
50,1.750368
60,1.438918
70,1.071876
80,0.837844
90,0.616485
100,0.482832


C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=300, training_loss=0.5222477483749389, metrics={'train_runtime': 13216.6599, 'train_samples_per_second': 0.182, 'train_steps_per_second': 0.023, 'total_flos': 323442612633600.0, 'train_loss': 0.5222477483749389, 'epoch': 3.0})

In [11]:
model.save_pretrained("ticket_lora_adapter")
tokenizer.save_pretrained("ticket_lora_adapter")

('ticket_lora_adapter\\tokenizer_config.json',
 'ticket_lora_adapter\\tokenizer.json')

In [12]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

# ---- paths ----
MODEL_PATH = "ticket_lora_adapter"   # folder where you saved the trained model
BASE_MODEL = "distilbert-base-uncased"

label_list = [
    "billing", "account", "orders", "shipping",
    "refund", "technical", "product", "subscription",
]
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

# ---- load tokenizer + model from saved folder ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model.eval()

# ---- prediction function ----
def predict(subject: str, body: str):
    text = f"{subject} {body}"
    inputs = tokenizer(text, truncation=True, padding="max_length", max_length=128, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        confidence, pred_id = torch.max(probs, dim=1)

    category = model.config.id2label[pred_id.item()]
    return category, confidence.item()

# ---- run on sample tickets ----
sample_tickets = [
    ("Payment failed", "My card was charged twice for the same order and I need a refund."),
    ("Can't login", "I keep entering my password correctly but it says invalid credentials."),
    ("Package never arrived", "Tracking says delivered but I never received my order."),
    ("App keeps crashing", "The mobile app crashes every single time I try to open the dashboard."),
    ("Cancel my subscription", "I want to cancel my subscription before the next billing cycle."),
]

print(f"{'Subject':30} | {'Predicted Category':20} | Confidence")
print("-" * 70)
for subject, body in sample_tickets:
    category, confidence = predict(subject, body)
    print(f"{subject:30} | {category:20} | {confidence:.2%}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Subject                        | Predicted Category   | Confidence
----------------------------------------------------------------------
Payment failed                 | billing              | 81.30%
Can't login                    | account              | 99.14%
Package never arrived          | shipping             | 94.64%
App keeps crashing             | technical            | 99.09%
Cancel my subscription         | subscription         | 87.75%


In [14]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

base_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=8)
model = PeftModel.from_pretrained(base_model, "ticket_lora_adapter")
tokenizer = AutoTokenizer.from_pretrained("ticket_lora_adapter")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [23]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="./support_tickets_balanced_800.csv")

# dataset["train"] currently holds everything — split it
split_dataset = dataset["train"].train_test_split(test_size=0.15, seed=42)

# Now you have both splits
dataset = split_dataset  # dataset["train"] and dataset["test"]

In [24]:
tokenized = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/680 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

In [26]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv("./support_tickets_balanced_800.csv")
train_df, test_df = train_test_split(df, test_size=0.15, stratify=df["category"], random_state=42)
train_df.to_csv("tickets_train.csv", index=False)
test_df.to_csv("tickets_test.csv", index=False)

In [27]:
dataset = load_dataset("csv", data_files={"train": "tickets_train.csv", "test": "tickets_test.csv"})

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [28]:
predictions = trainer.predict(tokenized["test"])

C:\Users\PRASHANT\AppData\Roaming\Python\Python311\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [29]:
predictions

PredictionOutput(predictions=array([[-2.57693124e+00, -3.61453938e+00, -2.82640457e-01,
        -2.54569459e+00, -2.42048788e+00, -5.56389093e-01,
         6.64394474e+00, -2.08031273e+00],
       [ 6.36607027e+00, -1.35928786e+00, -2.32576942e+00,
        -1.03415525e+00, -1.16430056e+00, -3.58665943e+00,
        -2.81300783e+00, -2.24178767e+00],
       [ 5.61502647e+00, -1.88695645e+00, -3.22170186e+00,
        -2.60306311e+00,  1.27146721e+00, -2.60133219e+00,
        -2.11797142e+00, -1.61045694e+00],
       [ 1.28266001e+00,  5.88891125e+00, -2.75305700e+00,
        -2.28754997e+00, -3.71962500e+00, -1.04625952e+00,
        -4.05266619e+00, -2.79348803e+00],
       [-1.61339331e+00,  5.14072514e+00, -1.13996696e+00,
        -2.55547571e+00, -3.99301195e+00, -1.65024668e-01,
        -1.68633914e+00, -2.35006309e+00],
       [ 5.60389996e-01, -3.05902529e+00, -2.04520750e+00,
        -2.93473721e+00,  6.71820736e+00, -2.58821559e+00,
        -1.88038719e+00, -2.25188851e+00],
     

In [30]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

In [31]:
from sklearn.metrics import classification_report, accuracy_score

acc = accuracy_score(labels, preds)
print(f"Accuracy: {acc:.2%}")

print(classification_report(labels, preds, target_names=label_list))

Accuracy: 100.00%
              precision    recall  f1-score   support

     billing       1.00      1.00      1.00        11
     account       1.00      1.00      1.00        20
      orders       1.00      1.00      1.00        13
    shipping       1.00      1.00      1.00        19
      refund       1.00      1.00      1.00        13
   technical       1.00      1.00      1.00        13
     product       1.00      1.00      1.00        17
subscription       1.00      1.00      1.00        14

    accuracy                           1.00       120
   macro avg       1.00      1.00      1.00       120
weighted avg       1.00      1.00      1.00       120

